<a href="https://colab.research.google.com/github/mcjkurz/qhchina/blob/main/tutorials/Intro_to_Python_for_Chinese_Humanities_Part_3_Collocation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Python for Chinese Humanities: Part 3, Collocation Analysis

**Welcome to Part 3 of the Tutorial!**

In this tutorial, we will work with Chinese texts and discover words which *attract* each other more than expected!

First, let's create a directory (文件夹) called "data," in which we will store our files. We can use the `mkdir` command to do that. Then, we will download Mao Zedong's *Selected Works* as our dataset (one .txt file).

In [1]:
!mkdir data
!wget https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/%E6%AF%9B%E6%B3%BD%E4%B8%9C%E9%80%89%E9%9B%86.txt -P data

--2024-10-08 08:59:09--  https://raw.githubusercontent.com/mcjkurz/qhchina/refs/heads/main/tutorials/data/%E6%AF%9B%E6%B3%BD%E4%B8%9C%E9%80%89%E9%9B%86.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5576316 (5.3M) [text/plain]
Saving to: ‘data/毛泽东选集.txt’

毛泽东选集.txt      100%[===================>]   5.32M  --.-KB/s    in 0.09s   

2024-10-08 08:59:09 (59.2 MB/s) - ‘data/毛泽东选集.txt’ saved [5576316/5576316]



We will now read this file and split it into lines using the `.split("\n")` method. The `\n` is a "newline character," also called "escape sequence." We will select only the lines that are at least 20 characters long, to remove all empty lines and short titles.

In [4]:
file_reader = open("data/毛泽东选集.txt", "r", encoding='utf-8')
text = file_reader.read()
file_reader.close()

long_lines = [line.strip() for line in text.split("\n") if len(line) > 20]
len(long_lines)

10213

We now need to load the tokenizer from the `spacy` library. We will disable certain parts of the `nlp` pipeline, such as `"ner"` (named entity recognition) or `"parser"` (for syntactic dependencies), to make word segmentation run faster.

In [3]:
!python -m spacy download zh_core_web_lg
import spacy
nlp = spacy.load("zh_core_web_lg")
nlp.select_pipes(disable=["ner", "senter", "parser"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.0/603.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


['ner', 'parser']

Now, we will take each line from the list `long_lines` and segment it into individual words using the `nlp` module.

Since we are interested in words only, we will remove punctuation, symbols, and spaces (if any).

The following code may take some time to finish (around 2 minutes). While there are ways to speed this up with `nlp.pipe()`, we will not do that for the sake of clarity.

In [5]:
from tqdm.auto import tqdm

excluded_tags = ['PUNCT', 'SYM', 'SPACE']

segmented_lines = []

for line in tqdm(long_lines):
  segmented_line = [token.text for token in nlp(line) if token.pos_ not in excluded_tags]
  segmented_lines.append(segmented_line)

len(segmented_lines)

  0%|          | 0/10213 [00:00<?, ?it/s]

10213

Let's see what the first 10 tokens from the 6th segmented line are.

In [6]:
segmented_lines[5][:10]

['无产阶级', '现代', '工业', '无产阶级', '约', '二百万', '人', '中国', '因', '经济']

# Building a co-occurrence matrix
We will now build a vocabulary (a list of unique words) from our corpus. We can do that using the class `Counter` from the `collections` library.

We will iterate over all `segmented_lines`, one by one, and update our `token_counter` at each iteration.

At the end, we will print the list of 10 most frequent words.

In [7]:
from collections import Counter
token_counter = Counter()

for line in segmented_lines:
  token_counter.update(line)

token_counter.most_common(10)

[('的', 69006),
 ('是', 16661),
 ('了', 13982),
 ('在', 12965),
 ('和', 12550),
 ('不', 10017),
 ('我们', 8359),
 ('有', 6896),
 ('要', 5758),
 ('中国', 5757)]

How many unique words are there?

In [12]:
len(token_counter.keys())

38474

We will choose only the top 10,000 most common words to reduce memory requirements.

We will then create two dictionaries, assigning a distinct ID to each unique word. In other words, there will be 10,000 IDs, from 0 to 9999. Each of them will correspond with a different word. We will be able to switch between IDs and words easily, which will become very useful for building a co-occurrence matrix.

In [25]:
vocab_size = 10000
vocab = [token for (token, count) in token_counter.most_common(vocab_size)] # the top 10k words

id2word = {i: vocab[i] for i in range(len(vocab))}
word2id = {vocab[i]: i for i in range(len(vocab))}

vocab = set(vocab)

print(word2id["中国"]) # what is the ID of the word 中国?
print(id2word[533]) # what is the word with ID 533?
print(id2word[word2id["中国"]]) # what is the word with ID word2id["中国"]?

9
政治局
中国


Now, let's build our co-occurrence matrix. We will adopt a **sliding window** approach, considering as collocates of the central words only the words that occur on the left and on the right side of this word within a specific horizon (distance).

Let's set `horizon = 2`.

For the sentence 今天 天气 那么 好 我们 去 散步 好 吗, the collocates of the word `"我们"` are `["那么", "好", "去", "散步"].`

This list is the `context` of `"我们"`.

In [26]:
import numpy as np

cooc_matrix = np.zeros((vocab_size, vocab_size)) # first create a matrix of zeros of the shape vocab_size x vocab_size

horizon = 2 # set the horizon (distance to the left and to the right)

for line in tqdm(segmented_lines): # for each segmented line
  for central_index in range(0, len(line)): # for each word

    central_word = line[central_index]
    if central_word not in vocab:
      continue # if the central word is not in our vocabulary, ignore it
    central_word_id = word2id[central_word]

    start_index = central_index - horizon # leftmost index
    if start_index < 0:
      start_index = 0

    end_index = central_index + horizon + 1 # rightmost index
    if end_index > len(line):
      end_index = len(line)

    # the context includes words on the left and on the right; the central word is excluded
    context = line[start_index:central_index] + line[central_index+1:end_index]

    for context_word in context:
      if context_word not in vocab:
        continue # if context word is not in the vocabulary, ignore it
      context_word_id = word2id[context_word]

      cooc_matrix[central_word_id, context_word_id] += 1 # finally, add 1 to the cooccurrence matrix

  0%|          | 0/10213 [00:00<?, ?it/s]

Great, we have built a word co-occurrence matrix! This will be very useful for our last step.

We will first choose an interesting word, called `target_word` here. For each word in the vocabulary, we will now build a contingency matrix and use Fisher's exact test to determine if this word is a statistically significant collocate of the chosen target word.

In [27]:
import scipy.stats as stats

target_word = "中国"
target_word_id = word2id[target_word]

# We can pre-compute some sums outside of the loop to save time
target_row_sum = sum(cooc_matrix[target_word_id, :])
entire_table_sum = cooc_matrix.sum()

collocates_data = []

for word in tqdm(vocab):
  word_id = word2id[word]
  word_row_sum = sum(cooc_matrix[word_id, :])

  a = cooc_matrix[target_word_id, word_id] * 2
  b = target_row_sum * 2 - a
  c = word_row_sum * 2 - a
  d = entire_table_sum - (a+b+c)

  contingency_table = [[a,b], [c,d]]
  odds, p_value = stats.fisher_exact(contingency_table, alternative="greater")

  collocate_info = {"word": word,
                    "obs": int(cooc_matrix[target_word_id, word_id]),
                    "p_value": p_value}
  collocates_data.append(collocate_info)

collocates_data.sort(key=lambda x: x["p_value"])
collocates_data[:20]

  0%|          | 0/10000 [00:00<?, ?it/s]

[{'word': '共产党', 'obs': 695, 'p_value': 0.0},
 {'word': '人民', 'obs': 847, 'p_value': 0.0},
 {'word': '革命', 'obs': 582, 'p_value': 2.9198604733470136e-177},
 {'word': '现阶段', 'obs': 38, 'p_value': 1.3811302841397366e-57},
 {'word': '境内', 'obs': 32, 'p_value': 2.7687461058261985e-49},
 {'word': '全', 'obs': 95, 'p_value': 4.0440091387117982e-44},
 {'word': '解放军', 'obs': 82, 'p_value': 1.0561372579160852e-40},
 {'word': '志愿军', 'obs': 22, 'p_value': 1.8564393702434722e-38},
 {'word': '卷', 'obs': 77, 'p_value': 6.014225195685952e-38},
 {'word': '领土', 'obs': 33, 'p_value': 3.021721994645096e-37},
 {'word': '访问', 'obs': 28, 'p_value': 3.304071967943083e-34},
 {'word': '侵略', 'obs': 59, 'p_value': 3.151665025668758e-32},
 {'word': '解放区', 'obs': 72, 'p_value': 4.48370905878581e-30},
 {'word': '新', 'obs': 127, 'p_value': 1.1421752864673074e-24},
 {'word': '身分', 'obs': 13, 'p_value': 2.0255070722162262e-23},
 {'word': '内政', 'obs': 17, 'p_value': 1.1104207490786116e-22},
 {'word': '哲学史', 'obs': 9, 'p

Now, let's save this in Excel format. We only want to keep the words with *p value* less than 0.05 (= statistically significant collocates).

In [28]:
# save collocates_data to excel
import pandas as pd
significant_collocates = [word for word in collocates_data if word["p_value"] < 0.05]

df = pd.DataFrame(significant_collocates)

save_filename = target_word + ".xlsx"
df.to_excel(save_filename, index=False)

You can now download the .xlsx file. **Congratulations**!